# Deney Sonuçları ve Değerlendirme

Bu bölümde, Yöntem 1 (Custom CNN) ve Yöntem 2 (Transfer Learning) modellerinin eğitim süreçleri, performans metrikleri (Accuracy, Loss, Precision, Recall, F1-Score) ve karmaşıklık matrisleri sunulmaktadır. Ayrıca, gerçek zamanlı demo sırasında elde edilen Kare Başına Saniye (FPS) değerleri de dahil edilmiştir.

In [2]:
# =============================================================================
# Gerekli Kütüphaneler
# =============================================================================
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# =============================================================================
# BÖLÜM 6: Modellerin Test Performansının İncelenmesi
# =============================================================================
# Eğitim bittiğine göre, şimdi sıra modellerimizin daha önce hiç görmediği
# test verileriyle nasıl bir performans sergilediğini görmekte.

print("\n" + "="*80)
print("Model Değerlendirme Aşaması Başlıyor...")
print("="*80)

# Değerlendireceğimiz en iyi modeller
models_to_evaluate = {
    "SVM": best_svm_model,
    "Random Forest": best_rf_model,
}

# Sonuçları biriktireceğimiz bir liste
evaluation_results = []

# Her bir modeli tek tek ele alalım
for model_name, model in models_to_evaluate.items():
    print(f"\n\n{'='*30} Şimdi Sırada: {model_name} Modeli {'='*30}")
    
    # Modelin test verisi üzerindeki tahminlerini alalım.
    # Eğitimde olduğu gibi, PCA'den geçmiş test verisini kullanıyoruz.
    if 'X_test_pca' not in locals():
        print("Uyarı: 'X_test_pca' bulunamadı. Bu, test performansını etkileyebilir.")
        # Fallback to a non-PCA version if needed, or handle the error
        continue

    print(f"Test verisi ({len(y_test)} örnek) üzerinde tahminler yapılıyor...")
    start_time = time.time()
    y_pred = model.predict(X_test_pca)
    end_time = time.time()
    prediction_time = end_time - start_time
    print(f"Tahmin işlemi {prediction_time:.4f} saniye sürdü.")

    # --- Detaylı Sonuç Raporu ---
    print("\n--- Modelin Karnesi: Sınıflandırma Raporu ---")
    report_dict = classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df['support'] = report_df['support'].astype(int)
    print(report_df.round(3))
    
    # Önemli metrikleri daha sonra karşılaştırmak için saklayalım
    accuracy = report_dict['accuracy']
    macro_f1 = report_dict['macro avg']['f1-score']
    weighted_f1 = report_dict['weighted avg']['f1-score']
    evaluation_results.append({
        "Model": model_name,
        "Doğruluk (%)": accuracy * 100,
        "Macro F1-Skoru": macro_f1,
        "Ağırlıklı F1-Skoru": weighted_f1,
        "Tahmin Süresi (s)": prediction_time
    })

    # --- Hata Matrisi ---
    print("\n--- Hataların Analizi: Karışıklık Matrisi ---")
    print("Bu matris, modelin hangi duyguları birbiriyle karıştırdığını gösterir.")
    cm = confusion_matrix(y_test, y_pred, labels=CLASS_NAMES)
    
    plt.figure(figsize=(12, 9))
    sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'Karışıklık Matrisi - {model_name}', fontsize=16)
    plt.xlabel('Modelin Tahmini', fontsize=12)
    plt.ylabel('Gerçek Sınıf', fontsize=12)
    plt.show()

    # --- Ek Analizler ---
    print("\n--- Daha Derin Bir Bakış: Ek Analizler ---")
    
    # Sınıfların kendi içindeki başarı oranları
    class_accuracy = cm.diagonal() / (cm.sum(axis=1) + 1e-6)
    class_accuracy_df = pd.DataFrame({
        'Duygu Sınıfı': CLASS_NAMES,
        'Başarı Oranı (%)': class_accuracy * 100
    }).sort_values(by='Başarı Oranı (%)', ascending=True)
    print("\na) Her bir duygu sınıfı için başarı oranları:")
    print(class_accuracy_df.to_string(index=False))

    # Modelin en çok zorlandığı dersler :)
    f1_scores = {cls: report_dict[cls]['f1-score'] for cls in CLASS_NAMES}
    worst_classes = sorted(f1_scores, key=f1_scores.get)[:3]
    print(f"\nb) Modelin en çok zorlandığı 3 sınıf: {', '.join(worst_classes)}")
    
    # Modelin en sık yaptığı hatalar
    np.fill_diagonal(cm, 0)
    errors = []
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            if i != j and cm[i, j] > 0:
                errors.append((CLASS_NAMES[i], CLASS_NAMES[j], cm[i, j]))
    
    errors.sort(key=lambda x: x[2], reverse=True)
    print("\nc) En sık yapılan 3 hata:")
    for i, (true_label, pred_label, count) in enumerate(errors[:3]):
        print(f"  {i+1}. '{true_label}' ifadesini {count} kez '{pred_label}' olarak tahmin etti.")


# =============================================================================
# BÖLÜM 7: Final Karşılaştırması
# =============================================================================

print("\n\n" + "="*80)
print("Nihai Karşılaştırma: Hangi Model Daha Başarılı?")
print("="*80)

if evaluation_results:
    df_final_results = pd.DataFrame(evaluation_results)
    # F1-Skoruna göre sıralayalım
    df_final_results_sorted = df_final_results.sort_values(by="Macro F1-Skoru", ascending=False)
    
    print("İki modelin test seti üzerindeki genel performans tablosu:")
    print(df_final_results_sorted.round(4).to_string(index=False))
    
    best_model_name = df_final_results_sorted.iloc[0]['Model']
    print(f"\nSonuç olarak, bu veri seti üzerinde en dengeli ve başarılı performansı **{best_model_name}** modeli gösterdi.")
    
    # En iyi modeli daha sonraki adımlar için ayıralım
    final_best_model = models_to_evaluate[best_model_name]
    print(f"En iyi model, 'final_best_model' değişkenine kaydedildi.")

else:
    print("Değerlendirme yapılabilecek bir model sonucu bulunamadı.")


Model Değerlendirme Aşaması Başlıyor...


NameError: name 'best_svm_model' is not defined

In [ ]:
print("BÖLÜM: KAMERA İLE OTOMATİK YÜZ TANIMA VE TAHMİN")


import os
import time

# Haar Cascade yolunu tam belirtelim
FACE_CASCADE_PATH = os.path.join(os.getcwd(), 'haarcascade_frontalface_default.xml')

face_cascade = cv2.CascadeClassifier(FACE_CASCADE_PATH)

if face_cascade.empty():
    print(f"HATA: Haar Cascade XML dosyası ('{FACE_CASCADE_PATH}') bulunamadı. Kamera testi yapılamıyor.")
else:
    CUSTOM_CNN_MODEL_PATH = 'best_custom_model.h5'
    demo_model = None
    
    if 'loaded_custom_cnn_model' in locals() and loaded_custom_cnn_model is not None:
        demo_model = loaded_custom_cnn_model
        print("Demo için Custom CNN modeli (bellekten) yüklendi.")
    else:
        try:
            demo_model = load_model(CUSTOM_CNN_MODEL_PATH)
            print(f"Demo için '{CUSTOM_CNN_MODEL_PATH}' modeli diskten yüklendi.")
        except Exception as e:
            print(f"Model yüklenirken hata oluştu: {e}")

    if demo_model is None:
        print("Model yüklenemediği için kamera demosu atlanıyor.")
    else:
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("HATA: Kamera açılamadı. Lütfen kameranızın bağlı ve çalışır durumda olduğundan emin olun.")
        else:
            prev_frame_time = 0
            
            while True:
                ret, frame = cap.read()
                if not ret:
                    print("Kareden okuma başarısız oldu, çıkılıyor...")
                    break

                new_frame_time = time.time()
                time_diff = new_frame_time - prev_frame_time
                fps = 1 / time_diff if time_diff > 0 else 0
                prev_frame_time = new_frame_time

                fps_text = f"FPS: {int(fps)}"
                cv2.putText(frame, fps_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                            1, (0, 255, 0), 2)

                gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

                faces = face_cascade.detectMultiScale(
                    gray_frame,
                    scaleFactor=1.1,
                    minNeighbors=5,
                    minSize=(30, 30)
                )

                for (x, y, w, h) in faces:
                    cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)

                    face_roi = gray_frame[y:y+h, x:x+w]
                    resized_face = cv2.resize(face_roi, MODEL_IMG_SIZE)

                    input_face = np.stack([resized_face]*3, axis=-1)
                    input_face = np.expand_dims(input_face, axis=0) / 255.0

                    predictions = demo_model.predict(input_face, verbose=0)
                    emotion_index = np.argmax(predictions[0])
                    predicted_emotion_label = CLASS_NAMES[emotion_index]
                    confidence = predictions[0][emotion_index] * 100

                    text = f"{predicted_emotion_label}: {confidence:.2f}%"
                    cv2.putText(frame, text, (x, y - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

                cv2.imshow('Real-Time Emotion Recognition', frame)

                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

            cap.release()
            cv2.destroyAllWindows()



BÖLÜM: KAMERA İLE OTOMATİK YÜZ TANIMA VE TAHMİN
Demo için Custom CNN modeli (bellekten) yüklendi.
